In [9]:
import os
from pyspark.sql import SparkSession

# -------------------------
# Environment
# -------------------------
os.environ["HADOOP_USER_NAME"] = "root"

spark = SparkSession.builder \
    .appName("snowflake_load") \
    .master("yarn") \
    .getOrCreate()

print("Spark Started")
# -------------------------
# Snowflake Config
# -------------------------
sf_options = {
    "sfURL": "jqnriwy-mb42464.snowflakecomputing.com",
    "sfUser": "Hanan",
    "sfPassword": "T4zna7CsFXZbbSB",
    "sfDatabase": "HEALTHCARE_DATA_DB",
    "sfSchema": "GOLD_LAYER",
    "sfWarehouse": "HEALTHCARE_WH",
    "sfRole": "SYSADMIN",

    "db": "HEALTHCARE_DATA_DB",
    "schema": "GOLD_LAYER",
    "warehouse": "HEALTHCARE_WH"
}

# -------------------------
# Path
# -------------------------
GOLD_BASE_PATH = "hdfs://hadoop-namenode:9000/user/root/datalake/gold/"

# -------------------------
# Load Function
# -------------------------
def load_table(table_name, mode="overwrite"):
    
    path = f"{GOLD_BASE_PATH}{table_name}"
    print(f"Reading: {path}")
    
    df = spark.read.parquet(path)

    # -------------------------
    # Optional: clean column names for Snowflake
    # -------------------------
    for col in df.columns:
        df = df.withColumnRenamed(col, col.upper())

    print(f"Loading {table_name} to Snowflake...")

    df.write \
    .format("net.snowflake.spark.snowflake") \
    .options(**sf_options) \
    .option("dbtable", table_name.upper()) \
    .option("usestagingtable", "off") \
    .mode(mode) \
    .save()

    print(f"SUCCESS: {table_name}")

# -------------------------
# RUN PIPELINE
# -------------------------
try:
    # DIMENSIONS (updated schema)
    load_table("dim_patient")
    load_table("dim_hospital")
    load_table("dim_condition")

    # FACT (UPDATED - includes new transformations)
    load_table("fact_healthcare")

    print("ALL TABLES LOADED SUCCESSFULLY")

except Exception as e:
    print(f"FAILED: {e}")

finally:
    spark.stop()

Spark Started
Reading: hdfs://hadoop-namenode:9000/user/root/datalake/gold/dim_patient
Loading dim_patient to Snowflake...
SUCCESS: dim_patient
Reading: hdfs://hadoop-namenode:9000/user/root/datalake/gold/dim_hospital
Loading dim_hospital to Snowflake...
SUCCESS: dim_hospital
Reading: hdfs://hadoop-namenode:9000/user/root/datalake/gold/dim_condition
Loading dim_condition to Snowflake...
SUCCESS: dim_condition
Reading: hdfs://hadoop-namenode:9000/user/root/datalake/gold/fact_healthcare
Loading fact_healthcare to Snowflake...
SUCCESS: fact_healthcare
ALL TABLES LOADED SUCCESSFULLY
